# Concise Implementation of Recurrent Neural Networks
:label:`sec_rnn-concise`

Like most of our from-scratch implementations,
:numref:`sec_rnn-scratch` was designed 
to provide insight into how each component works.
But when you are using RNNs every day 
or writing production code,
you will want to rely more on libraries
that cut down on both implementation time 
(by supplying library code for common models and functions)
and computation time 
(by optimizing the heck out of these library implementations).
This section will show you how to implement 
the same language model more efficiently
using the high-level API provided 
by your deep learning framework.
We begin, as before, by loading 
*The Time Machine* dataset.


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
%matplotlib inline

import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

## [**Defining the Model**]

We define the following class
using the RNN implemented
by high-level APIs.


In [3]:
from utils.helper import Module

class RNN(Module):  
    """The RNN model implemented with high-level APIs."""
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.save_hyperparameters()
        self.rnn = nn.RNN(num_inputs, num_hiddens)

    def forward(self, inputs, H=None):
        return self.rnn(inputs, H)

Inheriting from the `RNNLMScratch` class in :numref:`sec_rnn-scratch`, 
the following `RNNLM` class defines a complete RNN-based language model.
Note that we need to create a separate fully connected output layer.


In [4]:
from utils.helper import RNNLMScratch

class RNNLM(RNNLMScratch):  
    """The RNN-based language model implemented with high-level APIs."""
    def init_params(self):
        self.linear = nn.LazyLinear(self.vocab_size)

    def output_layer(self, hiddens):
        return self.linear(hiddens).swapaxes(0, 1)

## Training and Predicting

Before training the model, let's [**make a prediction 
with a model initialized with random weights.**]
Given that we have not trained the network, 
it will generate nonsensical predictions.


In [5]:
from utils.helper import TimeMachine

data = TimeMachine(batch_size=1024, num_steps=32)
rnn = RNN(num_inputs=len(data.vocab), num_hiddens=32)
model = RNNLM(rnn, vocab_size=len(data.vocab), lr=1)
model.predict('it has', 20, data.vocab)

'it hasgsggeggeggeggeggegge'

Next, we [**train our model, leveraging the high-level API**].


In [7]:
from utils.helper import Trainer

trainer = Trainer(max_epochs=100, gradient_clip_val=1)
trainer.fit(model, data)

epoch 1, train loss 2.997052, val loss 2.843951
epoch 2, train loss 2.846205, val loss 2.790515
epoch 3, train loss 2.795915, val loss 2.731687
epoch 4, train loss 2.729816, val loss 2.651505
epoch 5, train loss 2.647400, val loss 2.569197
epoch 6, train loss 2.571945, val loss 2.500441
epoch 7, train loss 2.508711, val loss 2.444067
epoch 8, train loss 2.453151, val loss 2.399093
epoch 9, train loss 2.409029, val loss 2.365171
epoch 10, train loss 2.374207, val loss 2.347176
epoch 11, train loss 2.351616, val loss 2.314195
epoch 12, train loss 2.326511, val loss 2.307073
epoch 13, train loss 2.299244, val loss 2.278358
epoch 14, train loss 2.284176, val loss 2.269645
epoch 15, train loss 2.258441, val loss 2.246441
epoch 16, train loss 2.241149, val loss 2.289107
epoch 17, train loss 2.222039, val loss 2.226597
epoch 18, train loss 2.206565, val loss 2.220830
epoch 19, train loss 2.201211, val loss 2.213237
epoch 20, train loss 2.181652, val loss 2.197147
epoch 21, train loss 2.165682

Compared with :numref:`sec_rnn-scratch`,
this model achieves comparable perplexity,
but runs faster due to the optimized implementations.
As before, we can generate predicted tokens 
following the specified prefix string.


In [8]:
model.predict('it has', 20, data.vocab)

'it has and the proven and '

## Summary

High-level APIs in deep learning frameworks provide implementations of standard RNNs.
These libraries help you to avoid wasting time reimplementing standard models.
Moreover,
framework implementations are often highly optimized, 
  leading to significant (computational) performance gains 
  when compared with implementations from scratch.

